<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/Cubic_Crystal_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs the jupyter-notebook given [here](https://github.com/materialsvirtuallab/m3gnet/blob/main/examples/Cubic%20Crystal%20Test.ipynb) on Colab.

In [ ]:
!pip install m3gnet alignn

In [ ]:
import traceback
import warnings

import numpy as np
import pandas as pd
from pymatgen.core import Composition, Lattice, Structure
#from pymatgen.ext.matproj import MPRester
#Run the cell twice to avoid MPRester error
#from pymatgen.ext.matproj import _MPResterLegacy as MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from tqdm import tqdm

from m3gnet.models import Relaxer

for category in (UserWarning, DeprecationWarning):
    warnings.filterwarnings("ignore", category=category, module="tensorflow")
    warnings.filterwarnings("ignore", category=category, module="pymatgen")

In [ ]:
data = pd.read_html("http://en.wikipedia.org/wiki/Lattice_constant")[0]

In [ ]:
data

In [ ]:

data = data[
    ~data["Crystal structure"].isin(
        ["Hexagonal", "Wurtzite", "Wurtzite (HCP)", "Orthorhombic", "Tetragonal perovskite", "Orthorhombic perovskite"]
    )
]
data.rename(columns={"Lattice constant (Å)": "a (Å)"}, inplace=True)

data["a (Å)"] = data["a (Å)"].map(float)


additional_fcc = """10 Ne 4.43 54 Xe 6.20
13 Al 4.05 58 Ce 5.16
18 Ar 5.26 70 Yb 5.49
20 Ca 5.58 77 Ir 3.84
28 Ni 3.52 78 Pt 3.92
29 Cu 3.61 79 Au 4.08
36 Kr 5.72 82 Pb 4.95
38 Sr 6.08 47 Ag 4.09
45 Rh 3.80 89 Ac 5.31
46 Pd 3.89 90 Th 5.08"""

additional_bcc = """3 Li 3.49 42 Mo 3.15
11 Na 4.23 55 Cs 6.05
19 K 5.23 56 Ba 5.02
23 V 3.02 63 Eu 4.61
24 Cr 2.88 73 Ta 3.31
26 Fe 2.87 74 W 3.16
37 Rb 5.59 41 Nb 3.30"""
def add_new(str_, structure_type, df):
    tokens = str_.split()
    new_crystals = []
    for i in range(int(len(tokens) / 3)):
        el = tokens[3 * i + 1].strip()
        if el not in df["Material"].values:
            new_crystals.append([tokens[3 * i + 1], structure_type, float(tokens[3 * i + 2])])
    df2 = pd.DataFrame(new_crystals, columns=data.columns)
    return pd.concat([df, df2])


# data = add_new(additional_fcc, "FCC", data)
# data = add_new(additional_bcc, "BCC", data)
# data.set_index("Material", inplace=True)
print(data)

Setting 5 Angstrom lattice constant to all crystals

In [ ]:
MAPI_KEY_LEGACY = '' #Optional, keep your MAPI_LEGACY_KEY here
predicted = []
mp = []
initial_structs = []
final_m3gnet_structs = []

#mpr = MPRester(MAPI_KEY_LEGACY)

relaxer = Relaxer()  # This loads the default pre-trained model

# warnings.filterwarnings(action="ignore", category=UserWarning, module="tensorflow")
currently_m3gnet_failing = ['CsCl','CsI','Fe','Mo','W','V','Nb','Ta','Ne','CaVO3','Xe','Ar','Yb','Kr','Sr','Ac','Li','Na','Cs','K','Ba','Eu','Cr','Rb',"NC0.99"]

#for formula, v in tqdm(data.iterrows(), total=len(data)):
for ii,i in tqdm(data.iterrows(), total=len(data)):
  #M3gnet failing so skipping these
  formula=i['Material']
  if formula not in currently_m3gnet_failing:
    formula = formula.split()[0]
    c = Composition(formula)
    els = sorted(c.elements)
    #cs = v["Crystal structure"]
    cs = i["Crystal structure"]
    wiki = i['a (Å)']
    # We initialize all the crystals with an arbitrary lattice constant of 5 angstroms.
    if "Zinc blende" in cs:
        s = Structure.from_spacegroup("F-43m", Lattice.cubic(5), [els[0], els[1]], [[0, 0, 0], [0.25, 0.25, 0.75]])
    elif "Halite" in cs:
        s = Structure.from_spacegroup("Fm-3m", Lattice.cubic(5), [els[0], els[1]], [[0, 0, 0], [0.5, 0, 0]])
    elif "Caesium chloride" in cs:
        s = Structure.from_spacegroup("Pm-3m", Lattice.cubic(5), [els[0], els[1]], [[0, 0, 0], [0.5, 0.5, 0.5]])
    elif "Cubic perovskite" in cs:
        s = Structure(
            Lattice.cubic(5),
            [els[0], els[1], els[2], els[2], els[2]],
            [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5], [0.5, 0.5, 0], [0.0, 0.5, 0.5], [0.5, 0, 0.5]],
        )
    elif "Diamond" in cs:
        s = Structure.from_spacegroup("Fd-3m", Lattice.cubic(5), [els[0]], [[0.25, 0.75, 0.25]])
    elif "BCC" in cs:
        s = Structure(Lattice.cubic(5), [els[0]] * 2, [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]])
    elif "FCC" in cs:
        s = Structure(Lattice.cubic(5), [els[0]] * 4, [[0.0, 0.0, 0.0], [0.5, 0.5, 0], [0.0, 0.5, 0.5], [0.5, 0, 0.5]])
    else:
        predicted.append(0)
        mp.append(0)
        continue
    print(s)
    initial_structs.append(s)
    relax_results = relaxer.relax(s)

    final_structure = relax_results["final_structure"]
    final_m3gnet_structs.append(final_structure)
    predicted.append(final_structure.lattice.a)

    #break
    try:
        mids = mpr.get_entries(s.composition.reduced_formula)
        for i in mids:
            try:
                structure = mpr.get_structure_by_material_id(i.entry_id)
                sga = SpacegroupAnalyzer(structure)
                sga2 = SpacegroupAnalyzer(final_structure)
                if sga.get_space_group_number() == sga2.get_space_group_number():
                    conv = sga.get_conventional_standard_structure()
                    mp.append(conv.lattice.a)
                    break
            except Exception:
                pass
        else:
            raise RuntimeError
    except Exception:
        mp.append(0)
        traceback.print_exc()
    print('\nInitial,final,MP,Wiki,formula',5,final_structure.lattice.a,mp[-1],wiki)

In [ ]:
data_1 = data[~data["Material"].isin(currently_m3gnet_failing)]

In [ ]:
data_1

In [ ]:
data_1["MP a (Å)"] = mp
data_1["Predicted a (Å)"] = predicted
data_1["initial_structs"] = initial_structs
data_1["final_m3gnet_structs"] = final_m3gnet_structs

In [ ]:
mem=[]
for i,ii in data_1.iterrows():
    info={}
    info['material']=ii['Material']
    info['a'] = ii['a (Å)']
    info['Crystal structure']=ii['Crystal structure']
    info['atoms']=(pmg_to_atoms(ii['initial_structs'])).to_dict()
    info['ref']=ii['Ref.']
    #print(info)
    mem.append(info)
from jarvis.db.jsonutils import dumpjson
dumpjson(data=mem,filename='data_1.json')

In [ ]:
from sklearn.metrics import mean_absolute_error
print('MAE wrt Exp/Wiki data',mean_absolute_error(data_1['a (Å)'],data_1['Predicted a (Å)']))
print('MAE wrt MP data',mean_absolute_error(data_1['MP a (Å)'],data_1['Predicted a (Å)']))

In [ ]:
from jarvis.core.atoms import Atoms
from alignn.ff.ff import ForceField, default_path
from jarvis.core.atoms import pmg_to_atoms
model_path = default_path()
def optimize_atoms(atoms=[],model_path=model_path):
    atoms = pmg_to_atoms(atoms)
    print(atoms.composition.reduced_formula)
    ff = ForceField(
        jarvis_atoms=atoms,
        model_path=model_path,
        stress_wt=.3,
        force_mult_natoms=False
    )
    opt, en, fs = ff.optimize_atoms()
    return opt.lattice.a

In [ ]:
tqdm.pandas()
#slow
data_1['alignn_a']=data_1['initial_structs'].progress_apply(lambda x:optimize_atoms(atoms=x))

In [ ]:
pip freeze